In [1]:
import pandas as pd
from pathlib import Path
import calendar

In [2]:
db_path = Path(r"D:\ProyectoAnalisisElectrico\MedidasValorizadas")
to_save_path = Path(r"D:\ProyectoAnalisisElectrico\DiaPromedio\Mensuales")

In [3]:
df = pd.read_parquet(db_path / "2505" / "2505_medidas_horarias.parquet")

In [4]:
df.head()

,Hora,clave,nombre_barra,tension,Zona,Razon_Social,RUT,Nombre_Corto,tipo,medida_3_sum,medida_3_min,CMg[CLP/KWh]_mean,valorizado_CLP_sum,Fecha_Medicion_last
0,0,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-33.18,-8.68,70.097555,-2325.86282,2025-05-01 00:45:00
1,1,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-33.60,-8.82,68.540260,-2304.28312,2025-05-01 01:45:00
2,2,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-34.58,-8.96,63.913277,-2210.00833,2025-05-01 02:45:00
3,3,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-35.00,-9.38,66.770067,-2334.46393,2025-05-01 03:45:00
4,4,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,L_D,-33.18,-8.82,69.821867,-2316.51870,2025-05-01 04:45:00


In [3]:
def get_days_in_month(date_int):
    year = date_int // 100
    month = date_int % 100
    _, days = calendar.monthrange(2000 + year, month)
    return days

In [4]:
group_columns = [
    'clave',
    'nombre_barra',
    'tension',
    'Zona',
    'Razon_Social',
    'RUT',
    'Nombre_Corto',
    'Hora_Dia',
    'Año_Mes',
    'tipo'
]

In [5]:
agg_rules = {
    'medida_3_sum': ['mean', 'std', 'count'], 
    'medida_3_min': ["min"],
    'CMg[CLP/KWh]_mean': ['mean', 'std', 'count'],
    'valorizado_CLP_sum': ['mean', 'std', 'count']
}

In [6]:
import pandas as pd

for folder_date in db_path.iterdir():
    if not folder_date.is_dir():
        continue
        
    date_str = folder_date.name
    
    # Filtra meses anteriores a la fecha de corte
    if int(date_str) < 2505: 
        continue

    to_save_folder = to_save_path / f"{date_str}"
    # Obtiene los días del mes para verificar posteriormente si la serie está completa
    n_days = get_days_in_month(int(date_str))
    
    if to_save_folder.is_dir():
        print(f"Data for {date_str} already processed. Skipping...")
        continue  
        
    to_save_folder.mkdir(parents=True, exist_ok=True)
    parquet_file = folder_date / f"{date_str}_medidas_horarias.parquet"

    print(f"Starting {date_str}...")

    df = pd.read_parquet(parquet_file)  
    
    # Extrae la hora y el mes para calcular el comportamiento horario promedio del mes
    df["Fecha_Medicion_last"] = pd.to_datetime(df["Fecha_Medicion_last"], format="%Y-%m-%d %H:%M:%S")
    df["Hora_Dia"] = df["Fecha_Medicion_last"].dt.hour
    df["Año_Mes"] = df["Fecha_Medicion_last"].dt.to_period('M').dt.to_timestamp()

    groups = df.groupby(by=group_columns, sort=False) 
    groups_with_size = groups.size()

    # Condición de validez: debe haber exactamente un registro por cada día del mes para esa hora
    valid_groups_mask = (groups_with_size == n_days)
    
    # Calcula las estadísticas (promedio, desviación estándar, etc.) agrupadas por hora
    df_aggregated = groups.agg(agg_rules)
    
    # Aplana la jerarquía de las columnas resultantes de la agregación
    df_aggregated.columns = [f"{col[0]}_{col[1]}" if isinstance(col, tuple) and col[1] else col[0] for col in df_aggregated.columns]
    
    df_valid = df_aggregated[valid_groups_mask].reset_index()
    
    # Estandariza los nombres de las métricas calculadas para mayor claridad
    rename_dict = {
        'Hora_Dia': 'Hora',
        'medida_3_sum_mean': 'medida_mean',
        'medida_3_sum_std': 'medida_std',
        'medida_3_sum_count': 'medida_count',
        'medida_3_min_min': 'medida_min', 
        'CMg[CLP/KWh]_mean_mean': 'CMg[CLP/KWh]_mean',
        'CMg[CLP/KWh]_mean_std': 'CMg[CLP/KWh]_std',
        'CMg[CLP/KWh]_mean_count': 'CMg[CLP/KWh]_count',
        'valorizado_CLP_sum_mean': 'valorizado_CLP_mean',
        'valorizado_CLP_sum_std': 'valorizado_CLP_std',
        'valorizado_CLP_sum_count': 'valorizado_CLP_count'
    }
    df_valid = df_valid.rename(columns=rename_dict)
    
    # -------------------------------------------------------------------------
    # NUEVO: Cálculo de medida_total y porcentajes para df_valid
    # -------------------------------------------------------------------------
    # Identificamos las columnas de agrupación base (excluyendo 'Hora_Dia' para sumar las 24hrs)
    base_group_cols = [col for col in group_columns if col != 'Hora_Dia']
    
    # Calculamos el total diario usando transform('sum') para mantener la cantidad de filas
    df_valid['medida_total'] = df_valid.groupby(base_group_cols)['medida_mean'].transform('sum')
    
    # Calculamos porcentajes y reemplazamos divisiones por 0 (que generan inf o NaN) con 0
    df_valid['medida_mean_porcentual'] = (df_valid['medida_mean'] / df_valid['medida_total']) \
                                          .replace([float('inf'), float('-inf')], 0).fillna(0)
    df_valid['medida_std_porcentual'] = (df_valid['medida_std'] / df_valid['medida_total']) \
                                          .replace([float('inf'), float('-inf')], 0).fillna(0)
    # -------------------------------------------------------------------------

    # Exporta el perfil horario mensual válido (series completas)
    df_valid.to_parquet(to_save_folder / f"{date_str}_mean_month.parquet", engine="pyarrow", compression="snappy")

    # Aísla y exporta los grupos incompletos (que no tienen datos para todos los días del mes)
    invalid_groups_mask = ~valid_groups_mask
    if invalid_groups_mask.any(): 
        print(f"Issues detected in {date_str} with {parquet_file.name}. Generating error file.")
        
        df_invalid = df_aggregated[invalid_groups_mask].reset_index()
        df_invalid = df_invalid.rename(columns=rename_dict)
        
        # -------------------------------------------------------------------------
        # NUEVO: Cálculo de medida_total y porcentajes para df_invalid
        # -------------------------------------------------------------------------
        df_invalid['medida_total'] = df_invalid.groupby(base_group_cols)['medida_mean'].transform('sum')
        df_invalid['medida_mean_porcentual'] = (df_invalid['medida_mean'] / df_invalid['medida_total']) \
                                              .replace([float('inf'), float('-inf')], 0).fillna(0)
        df_invalid['medida_std_porcentual'] = (df_invalid['medida_std'] / df_invalid['medida_total']) \
                                              .replace([float('inf'), float('-inf')], 0).fillna(0)
        # -------------------------------------------------------------------------
        
        df_invalid.to_parquet(to_save_folder / f"{date_str}_errores_month.parquet", engine="pyarrow", compression="snappy")

print("\nProcess Finished.")

Starting 2505...
Starting 2506...
Starting 2507...
Starting 2508...
Starting 2509...
Starting 2510...
Starting 2511...
Starting 2512...
Starting 2601...
Starting 2602...
Starting 2603...
Starting 2604...

Process Finished.
